In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("changelinglab/easycall-dysarthria")

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

test_dataset = dataset["test"]

In [3]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="italian",
    task="transcribe"
)

In [4]:
best_ckpt = "./whisper-medium-lora/checkpoint-1488"

model = WhisperForConditionalGeneration.from_pretrained(best_ckpt).to("cuda")
model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=1024, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=1024, out_features=32, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=32, out_features=1024, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict

In [5]:
def normalize(text):
    return text.lower().strip()

In [6]:
results = []

for sample in tqdm(test_dataset):

    audio = sample["audio"]["array"]
    text = sample["text"]
    severity = sample["dysarthria_severity"]
    speaker = sample["speaker"]
    filename = sample["filename"]

    # prepare input
    input_features = processor.feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to("cuda")

    # generate
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_length=64
        )

    pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # store
    results.append({
        "filename": filename,
        "speaker": speaker,
        "severity": severity,
        "ground_truth": normalize(text),
        "prediction": normalize(pred_text),
    })

  0%|          | 0/5213 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 5213/5213 [1:17:00<00:00,  1.13it/s]


In [7]:
df = pd.DataFrame(results)

df.head()

,filename,speaker,severity,ground_truth,prediction
0,fc10_06_Zero.wav,fc10,0,zero,zero
1,fc10_06_Richiama.wav,fc10,0,richiama,richiama chrani.
2,fc10_06_Cella.wav,fc10,0,cella,cella?
3,fc10_06_Stop.wav,fc10,0,stop,stop!
4,fc10_06_Vai_nei_preferiti.wav,fc10,0,vai nei preferiti,vaj ne preferi čistiti.


In [8]:
df.to_csv("asr_predictions.csv", index=False)
df.to_json("asr_predictions.json", orient="records", indent=2)

print("Saved predictions!")

Saved predictions!


In [9]:
for sev in sorted(df["severity"].unique()):
    print(f"\n===== Severity {sev} =====")
    display(df[df["severity"] == sev].head(10))


===== Severity 0 =====


,filename,speaker,severity,ground_truth,prediction
0,fc10_06_Zero.wav,fc10,0,zero,zero
1,fc10_06_Richiama.wav,fc10,0,richiama,richiama chrani.
2,fc10_06_Cella.wav,fc10,0,cella,cella?
3,fc10_06_Stop.wav,fc10,0,stop,stop!
4,fc10_06_Vai_nei_preferiti.wav,fc10,0,vai nei preferiti,vaj ne preferi čistiti.
5,fc10_06_Vai_alla_pagina_principale.wav,fc10,0,vai alla pagina principale,vai alla pagina principale.
6,fc10_06_Attiva_vivavoce.wav,fc10,0,attiva vivavoce,attiva vivavoce
7,fc10_06_Chiama_emergenza.wav,fc10,0,chiama emergenza,chiama emergenza 1
8,fc10_06_Scorri_verso_l_alto.wav,fc10,0,scorri verso l alto,scorri verso l' alto
9,fc10_06_No.wav,fc10,0,no,no



===== Severity 1 =====


,filename,speaker,severity,ground_truth,prediction
414,f09_06_Chiamata.wav,f09,1,chiamata,chiamata
415,f09_06_Cella.wav,f09,1,cella,sezna.
416,f09_06_Quattro.wav,f09,1,quattro,quattro quattro quattro quattro quattro quattr...
417,f09_06_Nuovo_contatto.wav,f09,1,nuovo contatto,nuovo contatto.
418,f09_06_Chiudi_applicazione.wav,f09,1,chiudi applicazione,chiudi apropa seo.
419,f09_06_Indietro.wav,f09,1,indietro,indietro
420,f09_06_Vai_alla_pagina_principale.wav,f09,1,vai alla pagina principale,vai repetena principala.
421,f09_06_Scorri_verso_il_basso.wav,f09,1,scorri verso il basso,scorri во сапа.
422,f09_06_Termina_chiamata.wav,f09,1,termina chiamata,termina chiamata
423,f09_06_Apri_rubrica_3.wav,f09,1,apri rubrica 3,apri rubrica



===== Severity 2 =====


,filename,speaker,severity,ground_truth,prediction
1242,m10_03_Stop.wav,m10,2,stop,so.
1243,m10_03_Nuovo_contatto.wav,m10,2,nuovo contatto,sopravo contatto sve sve sve sve sve sve sve s...
1244,m10_03_Cinque.wav,m10,2,cinque,si
1245,m10_03_Zero.wav,m10,2,zero,zero
1246,m10_03_Deseleziona.wav,m10,2,deseleziona,sezora
1247,m10_03_Cancella_contatto.wav,m10,2,cancella contatto,quattro contatto contatto contatto contatto co...
1248,m10_03_Sezione.wav,m10,2,sezione,sezora
1249,m10_03_Cella.wav,m10,2,cella,celele
1250,m10_03_Seleziona.wav,m10,2,seleziona,selezionama
1251,m10_03_Apri_rubrica_3.wav,m10,2,apri rubrica 3,mavu li?



===== Severity 3 =====


,filename,speaker,severity,ground_truth,prediction
3113,m11_06_Zelo.wav,m11,3,zelo,trebo se prekovao da ne prekovao da ne prekova...
3114,m11_06_Uno.wav,m11,3,uno,buele buele buele buele buele buele buele buel...
3115,m11_06_Deseleziona.wav,m11,3,deseleziona,si.si.si.si.si.si.si.si.si.si.si.si.si.si.si.s...
3116,m11_06_Apri.wav,m11,3,apri,sopračite sve sve sve sve sve sve sve sve sve ...
3117,m11_06_Apri_rubrica_2.wav,m11,3,apri rubrica 2,traveller
3118,m11_06_Vai_nei_preferiti.wav,m11,3,vai nei preferiti,preferiti
3119,m11_06_Preferiti.wav,m11,3,preferiti,prestavljeni
3120,m11_06_Scendi.wav,m11,3,scendi,sredi
3121,m11_06_Sei.wav,m11,3,sei,sei
3122,m11_06_Scorri_verso_l_alto.wav,m11,3,scorri verso l alto,skojoj učin



===== Severity N/A =====


,filename,speaker,severity,ground_truth,prediction
1655,f04_04_Deseleziona.wav,f04,N/A,deseleziona,richiama di...
1656,f04_04_Otto.wav,f04,N/A,otto,sopračno
1657,f04_04_Muovi.wav,f04,N/A,muovi,nove
1658,f04_04_Quattro.wav,f04,N/A,quattro,quattro
1659,f04_04_Termina chiamata.wav,f04,N/A,termina chiamata,termina chiamata
1660,f04_04_Tre.wav,f04,N/A,tre,treći
1661,f04_04_Cinque.wav,f04,N/A,cinque,se uči
1662,f04_04_Top.wav,f04,N/A,top,stop
1663,f04_04_Nuovo contatto.wav,f04,N/A,nuovo contatto,nuovo contatto
1664,f04_04_Sotto.wav,f04,N/A,sotto,sjelatno


In [10]:
for sev in df["severity"].unique():
    df[df["severity"] == sev].to_csv(f"asr_severity_{sev}.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'asr_severity_N'